# 01 — Pipeline Walkthrough

End-to-end run of the LLM ETL & evaluation pipeline against the repo's demo data.

**Stages**

1. ETL — raw CSV/JSONL → canonical `llm_inputs.jsonl`
2. Prompting — config-driven system + user prompts
3. Simulator — deterministic noisy LLM outputs
4. Transform — parse, coerce, validate → accepted/rejected
5. Roundtrip — re-validate accepted records as API payloads
6. Evaluation — metrics + error taxonomy

Run this notebook from the repo root.

In [1]:
import json
import sys
from dataclasses import asdict
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src" / "llm_etl_eval").is_dir():
    sys.path.insert(0, str(REPO_ROOT / "src"))
elif (REPO_ROOT.parent / "src" / "llm_etl_eval").is_dir():
    REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_RAW = REPO_ROOT / "data" / "raw"
SCHEMA_PATH = REPO_ROOT / "data" / "expected" / "target_schema.json"
GOLD_PATH = REPO_ROOT / "data" / "gold" / "labels.jsonl"
BASELINE_CONFIG = REPO_ROOT / "configs" / "baseline.yaml"
WORK = REPO_ROOT / "notebooks" / "_work" / "01_walkthrough"
WORK.mkdir(parents=True, exist_ok=True)


def show(path, n=2):
    """Print first n JSONL records from a path."""
    for i, line in enumerate(path.read_text(encoding="utf-8").splitlines()):
        if i >= n or not line.strip():
            break
        print(json.dumps(json.loads(line), indent=2)[:600])
        print("---")


REPO_ROOT, WORK

(WindowsPath('C:/Users/diogo/work_code/llm-etl-and-evaluation'),
 WindowsPath('C:/Users/diogo/work_code/llm-etl-and-evaluation/notebooks/_work/01_walkthrough'))

## 1. ETL

Reads `tickets.csv` + `events.jsonl`, cleans, joins by `ticket_id`, normalizes, and emits canonical input records.

In [2]:
from llm_etl_eval.etl.run import run_pipeline

processed_dir = WORK / "processed"
run_pipeline(DATA_RAW, processed_dir)

inputs_path = processed_dir / "llm_inputs.jsonl"
print(f"Records emitted: {len(inputs_path.read_text(encoding='utf-8').splitlines())}")
show(inputs_path, n=1)

Records emitted: 12
{
  "actions": [
    {
      "action": "triage",
      "due_at": "2025-11-11T14:30:00+00:00",
      "owner": "AG-10",
      "status": "open"
    }
  ],
  "category": "billing",
  "created_at": "2025-11-10T14:30:00+00:00",
  "customer_id": "CUST-202",
  "entities": [
    {
      "confidence": 0.7,
      "type": "browser",
      "value": "Chrome"
    }
  ],
  "language": "en",
  "priority": "medium",
  "summary": "Summary unavailable",
  "ticket_id": "TKT-1000"
}
---


## 2. Prompting

Generates system + user prompt payloads per record. Strictness and schema reminders come from the config — same code path produces baseline (minimal) and improved (strict + schema) prompts.

In [3]:
from llm_etl_eval.prompting.run import run_prompting

prompts_path = WORK / "prompts.jsonl"
run_prompting(inputs_path, prompts_path, SCHEMA_PATH, BASELINE_CONFIG)

first = json.loads(prompts_path.read_text(encoding="utf-8").splitlines()[0])
print("SYSTEM:\n", first["system"][:400], "\n\n---\n\nUSER:\n", first["user"][:400])

SYSTEM:
 You are a data normalization assistant. Produce a JSON object that matches the target schema.

Rules:
- Output only JSON (no markdown, no commentary).
- Required fields: ticket_id, customer_id, created_at, language, summary, category, priority, entities, actions.
- Do not include additional fields. 

---

USER:
 {"task": "Normalize the ticket record to the target schema.", "input": {"actions": [{"action": "triage", "due_at": "2025-11-11T14:30:00+00:00", "owner": "AG-10", "status": "open"}], "category": "billing", "created_at": "2025-11-10T14:30:00+00:00", "customer_id": "CUST-202", "entities": [{"confidence": 0.7, "type": "browser", "value": "Chrome"}], "language": "en", "priority": "medium", "summary": "


## 3. Simulator

Deterministic LLM substitute: produces realistic-but-noisy outputs at a controlled error rate. Lets us test the full pipeline without paying for API calls.

In [4]:
from llm_etl_eval.sim.llm_simulator import (
    SimulationConfig, load_inputs, simulate_records, write_outputs,
)

outputs_path = WORK / "outputs.jsonl"
sim_config = SimulationConfig(error_rate=0.35, seed=7)
write_outputs(simulate_records(load_inputs(inputs_path), sim_config), outputs_path)

lines = outputs_path.read_text(encoding="utf-8").splitlines()
print(f"Outputs: {len(lines)}")
show(outputs_path, n=1)

Outputs: 12
{
  "actions": [
    {
      "action": "triage",
      "due_at": "2025-11-11T14:30:00+00:00",
      "owner": "AG-10",
      "status": "open"
    }
  ],
  "category": "billing",
  "created_at": "2025-11-10T14:30:00+00:00",
  "customer_id": "CUST-202",
  "entities": [
    {
      "confidence": 0.7,
      "type": "browser",
      "value": "Chrome"
    }
  ],
  "language": "en",
  "priority": "critical",
  "summary": "Summary unavailable",
  "ticket_id": "TKT-1000"
}
---


## 4. Transform

Tolerant parse → contract validation → coercion + mapping → schema validation. Splits records into `accepted.jsonl` (safe to ingest) and `rejected.jsonl` (with structured errors).

In [5]:
from llm_etl_eval.transform.run import run_transform

final_dir = WORK / "final"
run_transform(SCHEMA_PATH, outputs_path, final_dir, BASELINE_CONFIG)

accepted = (final_dir / "accepted.jsonl").read_text(encoding="utf-8").splitlines()
rejected = (final_dir / "rejected.jsonl").read_text(encoding="utf-8").splitlines()
print(f"accepted={len(accepted)} rejected={len(rejected)}")

accepted=8 rejected=4


## 5. Roundtrip

Proves accepted records would be accepted by a downstream API: validates them again and wraps them in mock payloads. Catches subtle drift between transform and target schema.

In [6]:
from llm_etl_eval.checks.roundtrip import roundtrip_accepteds

api_path = WORK / "api_payloads.jsonl"
payloads = roundtrip_accepteds(
    final_dir / "accepted.jsonl", SCHEMA_PATH, api_path,
)
print(f"API payloads written: {len(payloads)}")

API payloads written: 8


## 6. Evaluation

Computes the headline metrics — schema validity, completeness, field accuracy vs gold, constraint violations, consistency — plus an error taxonomy.

In [7]:
from llm_etl_eval.eval.metrics import compute_metrics

schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
metrics = compute_metrics(str(outputs_path), str(GOLD_PATH), schema)

print(f"records={metrics.totals['records']} gold={metrics.totals['gold_records']}")
print(f"schema_validity_rate    = {metrics.schema_validity_rate:.2%}")
print(f"completeness_rate       = {metrics.completeness_rate:.2%}")
print(f"field_accuracy          = {metrics.field_accuracy:.2%}")
print(f"constraint_violations/r = {metrics.constraint_violations_per_record:.2f}")
print(f"consistency_rate        = {metrics.consistency_rate:.2%}")
print("\nTop error codes:")
for code, count in list(metrics.error_taxonomy.items())[:5]:
    print(f"  {count:3d}  {code}")

records=12 gold=50
schema_validity_rate    = 66.67%
completeness_rate       = 97.22%
field_accuracy          = 28.33%
constraint_violations/r = 0.17
consistency_rate        = 100.00%

Top error codes:
    2  'critical' is not one of ['low', 'medium', 'high', 'urgent']
    2  'summary' is a required property


## Summary

Every stage is a thin, testable CLI. The full pipeline is reproducible: same config + same seed → same outputs.

Continue with [02_evaluation_metrics.ipynb](02_evaluation_metrics.ipynb) for a metrics deep-dive, or [03_monitoring_manifests_drift_clusters.ipynb](03_monitoring_manifests_drift_clusters.ipynb) for the monitoring story.